# Human-in-the-Loop: Pre-Action Gates, Risk Tiering, and Audit Logging

The README for dis lesson introduce Human-in-the-Loop wit a short snippet wey ask di user `APPROVE` or `REJECT` after di agent don already produce response. Dat pattern na correct starting point, but production HITL implementations usually need three additional tins:

1. A **pre-action gate** wey run **before** di agent carry out one risky step, so cost, irreversibility, and latency go dey under control.
2. **Risk tiering**, so low-risk actions auto-execute, medium-risk actions dem dey batch-approved, and only high-risk actions go block for human.
3. An **audit log plus revision loop**, so every gate decision dey recorded as JSONL, and if dem reject e, e go re-prompt di agent wit structured reason instead make e just print `Revising...`.

Dis notebook go build each of dis on top di same primitives as `06-system-message-framework.ipynb`. E go run end-to-end for `DEMO_MODE = True` (no need interactive input) or wit real `input()` prompts when `DEMO_MODE = False`. Note: inside DEMO_MODE di third goal retry na script so di loop mechanics go visible end-to-end. Real revision-driven re-classification need `DEMO_MODE = False` and operator.

**Out of scope (handled in other lessons):** authentication and access control (Lesson 06 README threat 2), tool-call middleware (Lesson 14 MAF deep dive), multi-agent debate patterns.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: Pre-action gate

The README's HITL snippet dey call the agent first, den e go ask the user make e approve the output. Na **post-action** flow be that one. The agent don already run, so the LLM call cost don already pay, and any side effect (email wey dem don send, database row wey dem don write, comment wey dem don post) don already happen.

A **pre-action** flow dey put the gate before the agent run the risky step. The agent go propose the action, the gate go decide whether to run am, and only if e approve na that time the side effect go happen.

| Aspect | Post-action approval (README snippet) | Pre-action gate (this notebook) |
|---|---|---|
| When does approval run? | After the agent don produce output | Before any side-effect run |
| LLM cost on rejection | Don already pay | Na only for the proposal e dey pay, no be for the action |
| Irreversible side effects | E fit happen (action don already run) | E dey prevent am |
| Audit clarity | Approval na print statement | Approval na JSON record with timestamp, action, reason |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Risk tiering

No every action need human approval. To check something wey be read-only for public API different from to send customer email get different risk. To treat all the same go waste operator attention and slow down the agent.

Simple 3-tier model:

| Tier | Examples | Approval flow |
|---|---|---|
| `low` (read-only) | Search knowledge base, check flight options, collect public web page | Auto-execute, logged for audit |
| `medium` (cheap mutation) | Cache result, increment counter, schedule reminder | Auto-execute, but batched daily review |
| `high` (external-facing or irreversible) | Send email, charge card, post to public channel | Block on human approval |

This na one kind tiering. Production systems dem dey use more detailed tiers (like AWS IAM permission levels, role-based access tiers). The 3-tier wey dey below na the smallest useful version for agent wey dey do read-only and side-effecting actions together.

The classifier below dey use keyword heuristics so demo go remain deterministic and cheap. For production system, you fit change am with learned classifier or policy engine.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pattern 3: Audit log and revision loop

A `print("Response approved.")` no be audit log. For trust, every gate decision suppose dey recorded as structured event wey you fit later query, replay, or attach to incident review.

Two parts:

1. **Append-only JSONL.** One line per decision, with timestamp, action, tier, decision, reason. E easy to grep, easy to send go real log store later.
2. **Revision loop on rejection.** When gate talk `deny`, the agent go re-prompts itself with the rejection reason inside context, so the next proposal fit avoid the wahala.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Additional resources

Plenty oda public projekts dem dey use diffren variation dem of dis HITL patern dem. Compare dem way dem dey do am to sabi weh go best for your stack:

- **LangChain** human-in-the-loop tool wrappers ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): tool wrapper wey you fit just drop inside wey go stop to wait for human input.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ don arrange dis one): e dey use agent role wey specifically sef represent human for multi-agent talk.
- **Semantic Kernel** function filters ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware wey dey run for any function call, e good for control logic gate.

Every project dey handle the three sub-patern dem different: LangChain dey wrap dem as tools, AutoGen dey use agent role, Semantic Kernel dey use middleware filters. Read one or two implementation well well before you choose design wey you go use for your own agent.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
